# Convert LoRA Adapters to Quantized GGUF

This notebook:
1. Loads your base model + LoRA adapters
2. Merges each adapter into the base model
3. Converts to GGUF format
4. Quantizes to Q4_K_M
5. Uploads to Hugging Face Hub

## 1. Install Dependencies

In [ ]:
!pip install torch transformers peft huggingface_hub sentencepiece protobuf

## 2. Clone and Build llama.cpp

In [ ]:
import os
import shutil

# Remove existing llama.cpp directory to ensure a clean clone
if os.path.exists('llama.cpp'):
    shutil.rmtree('llama.cpp')

!git clone https://github.com/ggerganov/llama.cpp
%cd llama.cpp
# Remove existing build directory to ensure a clean build
if os.path.exists('build'):
    shutil.rmtree('build')
!mkdir build
%cd build
!cmake ..
# Build only the llama-quantize target to speed up compilation
!cmake --build . --target llama-quantize --config Release
%cd ../..


Cloning into 'llama.cpp'...
remote: Enumerating objects: 76304, done.
remote: Counting objects: 100% (317/317), done.
remote: Compressing objects: 100% (258/258), done.
remote: Total 76304 (delta 178), reused 59 (delta 59), pack-reused 75987 (from 3)
Receiving objects: 100% (76304/76304), 280.02 MiB | 17.08 MiB/s, done.
Resolving deltas: 100% (55327/55327), done.
/content/llama.cpp
/content/llama.cpp/build
-- The C compiler identification is GNU 11.4.0
-- The CXX compiler identification is GNU 11.4.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
CMAKE_BUILD_TYPE=Release
-- Found Git: /usr/bin/git (found version "2.34.1")
-- The A

## 3. Login to Hugging Face

In [ ]:
from huggingface_hub import login, HfApi

# Login with your token (needs write access for uploading)
login()

## 4. Configuration

**Update these paths to match your setup:**

In [ ]:
import os

# Base model
BASE_MODEL = "Qwen/Qwen3-1.7B"

# Your LoRA adapter paths on HF Hub (or local paths)
# Update these to your actual adapter locations
ADAPTERS = {
    "en": "./English",  # Example for a local path
    "es": "./Spanish",
    "ko": "./Korean",
}

# Quantization type - Changed from Q4_K_M to Q5_K_M for potentially better quality
QUANT_TYPE = "Q5_K_M"

# output directory


# Local working directory
WORK_DIR = "./gguf_conversion"
os.makedirs(WORK_DIR, exist_ok=True)

## 5. Load Base Model

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

def load_base_model():
    """Load a fresh copy of the base model."""
    print("Loading base model...")
    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        dtype="auto",
        device_map="auto"
    )
    print("Base model loaded.")
    return model

# Load tokenizer once (reused across all adapters)
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
print("Tokenizer loaded.")

## 6. Merge and Convert Functions

In [ ]:
import subprocess
import shutil
from peft import PeftModel

def merge_adapter(adapter_path, output_path):
    """Merge LoRA adapter into a fresh base model."""
    # Load fresh base model to avoid PEFT multi-adapter issues
    base_model = load_base_model()
    
    print(f"  Loading adapter from {adapter_path}...")
    model = PeftModel.from_pretrained(
        base_model,
        adapter_path,
        is_trainable=False
    )

    print("  Merging weights...")
    merged = model.merge_and_unload()

    print(f"  Saving to {output_path}...")
    merged.save_pretrained(output_path, safe_serialization=True)
    tokenizer.save_pretrained(output_path)

    # Free memory
    del model, merged, base_model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return output_path


def convert_to_gguf(model_path, output_file):
    """Convert HF model to GGUF format."""
    print(f"  Converting to GGUF...")
    result = subprocess.run([
        "python", "llama.cpp/convert_hf_to_gguf.py",
        model_path,
        "--outfile", output_file,
        "--outtype", "f16"
    ], capture_output=True, text=True)

    if result.returncode != 0:
        raise RuntimeError(f"GGUF conversion failed: {result.stderr}")
    return output_file


def quantize_gguf(input_file, output_file, quant_type=QUANT_TYPE):
    """Quantize GGUF file."""
    print(f"  Quantizing to {quant_type}...")
    result = subprocess.run([
        "llama.cpp/build/bin/llama-quantize",
        input_file,
        output_file,
        quant_type
    ], capture_output=True, text=True)

    if result.returncode != 0:
        raise RuntimeError(f"Quantization failed: {result.stderr}")
    return output_file

## 7. Process All Adapters

In [ ]:
gguf_files = {}

for language, adapter_path in ADAPTERS.items():
    print(f"\n{'='*50}")
    print(f"Processing: {language}")
    print('='*50)

    # Paths
    merged_path = f"{WORK_DIR}/{language}-merged"
    f16_gguf = f"{WORK_DIR}/{language}-f16.gguf"
    quantized_gguf = f"{WORK_DIR}/{language}-{QUANT_TYPE.lower()}.gguf"

    try:
        # Step 1: Merge (loads fresh base model internally)
        merge_adapter(adapter_path, merged_path)

        # Step 2: Convert to GGUF
        convert_to_gguf(merged_path, f16_gguf)

        # Step 3: Quantize
        quantize_gguf(f16_gguf, quantized_gguf, QUANT_TYPE)

        gguf_files[language] = quantized_gguf

        # Show file size
        size_mb = os.path.getsize(quantized_gguf) / (1024 * 1024)
        print(f"  Created: {quantized_gguf} ({size_mb:.1f} MB)")

    except Exception as e:
        print(f"  ERROR processing {language}: {e}")
        raise  # Stop processing on failure

    finally:
        # Cleanup intermediate files
        if os.path.exists(f16_gguf):
            os.remove(f16_gguf)
        if os.path.exists(merged_path):
            shutil.rmtree(merged_path)

print(f"\n{'='*50}")
print("All conversions complete!")
print('='*50)

## 8. Upload to Hugging Face Hub

In [ ]:
from huggingface_hub import HfApi, create_repo

api = HfApi()

model_name_for_repo = BASE_MODEL.split('/')[-1]
OUTPUT_REPO = f"{model_name_for_repo}-{QUANT_TYPE.replace('_', '')}-language-LoRA"

# Create repo if it doesn't exist
try:
    create_repo(repo_id=OUTPUT_REPO, exist_ok=True, repo_type="model")
    print(f"Repository ready: {OUTPUT_REPO}")
except Exception as e:
    print(f"Note: {e}")

# Upload each GGUF file
for language, filepath in gguf_files.items():
    filename = os.path.basename(filepath)
    print(f"Uploading {filename}...")

    api.upload_file(
        path_or_fileobj=filepath,
        path_in_repo=filename,
        repo_id=OUTPUT_REPO,
        repo_type="model"
    )
    print(f"  Uploaded: {filename}")

print(f"\nAll GGUF files uploaded to: https://huggingface.co/{OUTPUT_REPO}")

In [ ]:
# Generate and upload a README.md (Model Card)

readme_content = f"""---
title: {OUTPUT_REPO}
author: Your Name/Organization
licenses: apache-2.0
tags:
- qwen
- lora
- gguf
- quantization
- adapter
---

Multilingual GGUF Models
Quantized GGUF models for CPU inference using llama.cpp.

Files
"""

# List the generated files
for language, filepath in gguf_files.items():
    filename = os.path.basename(filepath)
    readme_content += f"{language.lower()}: {filename}\n"

# Save the README locally
readme_path = os.path.join(WORK_DIR, "README.md")
with open(readme_path, "w") as f:
    f.write(readme_content)

# Upload the README.md to the Hugging Face Hub
print("Uploading README.md...")
api.upload_file(
    path_or_fileobj=readme_path,
    path_in_repo="README.md",
    repo_id=OUTPUT_REPO,
    repo_type="model"
)
print("  Uploaded: README.md")

print(f"\nView your models and model card here: https://huggingface.co/{OUTPUT_REPO}")

## Done!

Your GGUF models are now on Hugging Face Hub. Update your `main.py` to use them with `llama-cpp-python`.